In [1]:
import pandas as pd
import numpy as np

interaction = pd.read_parquet(
    "../data/processed/interaction_matrix.parquet"
)

interaction.shape

(8640, 252)

In [2]:
interaction.head()

skuNumber,SKU00001,SKU00009,SKU00010,SKU00020,SKU00022,SKU00024,SKU00026,SKU00030,SKU00033,SKU00046,...,SKUSA2,SKUSA3,SKUSA4,SKUSA5,SKUSA6,SKUSA7,SKUSA8,SKUSA9,SKUSY1,SKUTZ3
customerId,,,,,,,,,,,,,,,,,,,,,
USR-100,0.0,0.0,1.0,0.0,2.0,0.0,1.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
USR-1000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
USR-100021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
USR-100025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
USR-100062,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

In [4]:
similarity_matrix = cosine_similarity(
    interaction
)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=interaction.index,
    columns=interaction.index
)

similarity_df.shape

(8640, 8640)

In [5]:
sample_customer = interaction.index[0]

similar_users = (
    similarity_df[sample_customer]
    .sort_values(
        ascending=False
    )
)

similar_users.head(10)

customerId
USR-100       1.000000
USR-161606    0.940785
USR-74105     0.935293
USR-37490     0.935279
USR-13170     0.931028
USR-92704     0.928455
USR-37626     0.928264
USR-38175     0.927818
USR-38535     0.924860
USR-1519      0.922810
Name: USR-100, dtype: float64

In [6]:
def recommend_products(
    customer_id,
    top_n=5
):
    
    similar_users = (
        similarity_df[customer_id]
        .sort_values(
            ascending=False
        )[1:11]
    )

    customer_products = set(
        interaction.loc[
            customer_id
        ][
            interaction.loc[
                customer_id
            ] > 0
        ].index
    )

    recommendations = {}

    for user in similar_users.index:

        similarity_score = (
            similar_users[user]
        )

        purchased = (
            interaction.loc[user]
        )

        for sku in purchased[
            purchased > 0
        ].index:

            if sku not in customer_products:

                recommendations[
                    sku
                ] = (
                    recommendations.get(
                        sku,
                        0
                    )
                    +
                    similarity_score
                )

    recommendations = (
        sorted(
            recommendations.items(),
            key=lambda x: x[1],
            reverse=True
        )
    )

    return recommendations[:top_n]

In [7]:
sample_customer = (
    interaction.index[0]
)

recommend_products(
    sample_customer
)

[('SKU00075', np.float64(4.667438394987487)),
 ('SKUPM5', np.float64(2.799026187806736)),
 ('SKU00009', np.float64(2.791180622604111)),
 ('SKU613', np.float64(2.7823346803381876)),
 ('SKU00001', np.float64(1.866320374063244))]

In [8]:
sku_lookup = pd.read_parquet(
    "../data/processed/sku_lookup.parquet"
)

sku_dict = dict(
    zip(
        sku_lookup["skuNumber"],
        sku_lookup["itemName"]
    )
)

In [9]:
sample_customer = (
    interaction.index[0]
)

recs = recommend_products(
    sample_customer
)

for sku, score in recs:

    print(
        sku_dict.get(
            sku,
            sku
        ),
        round(score, 2)
    )

Tulsi Royal Khajoor 4.67
Rajnigandha 17g 2 Pcs 2.8
Chingles Maxi TF Jar + 1 Rs 5 Lollipop 2.79
Center Fresh 2.78
Chingles Filz Jar + 1 Rs.5 Lollipop 1.87


In [10]:
recommend_products(
    interaction.index[0]
)

[('SKU00075', np.float64(4.667438394987487)),
 ('SKUPM5', np.float64(2.799026187806736)),
 ('SKU00009', np.float64(2.791180622604111)),
 ('SKU613', np.float64(2.7823346803381876)),
 ('SKU00001', np.float64(1.866320374063244))]

In [11]:
for customer in interaction.index[:5]:

    print("\n")
    print("Customer:", customer)

    recs = recommend_products(
        customer
    )

    for sku, score in recs:

        print(
            sku_dict.get(
                sku,
                sku
            )
        )



Customer: USR-100
Tulsi Royal Khajoor
Rajnigandha 17g 2 Pcs
Chingles Maxi TF Jar + 1 Rs 5 Lollipop
Center Fresh
Chingles Filz Jar + 1 Rs.5 Lollipop


Customer: USR-1000
Pulse Kachha Aam 175 Candy
RG Pearl Elaichi 1g Hanger
Luvit Eclairs
Alpenliebe Gold Caramel Candy
Pass Pass Meetha Magic 11.75g Hanger


Customer: USR-100021


Customer: USR-100025


Customer: USR-100062
Pass Pass Meetha Magic 2.2g (100 pcs)
Rajnigandha 100g 
Tulsi 00 10g TIN
Act II Butter Popcorn
Rajnigandha 17g Zipper 1 Pcs


In [12]:
similarity_df.to_parquet(
    "../data/processed/similarity_matrix.parquet"
)

print("Saved")

Saved


In [13]:
interaction.shape

(8640, 252)

In [14]:
similarity_df.to_parquet(
    "../data/processed/similarity_matrix.parquet"
)

print("Similarity Matrix Saved")

Similarity Matrix Saved


In [15]:
similarity_df.shape

(8640, 8640)